In [ ]:
#####################################################################
#########   Code to download files at selected intervals    #########
#####################################################################

import time
import os
import csv
import glob
from datetime import datetime as dt_datetime, timedelta
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Define the path to the folder containing the CSV files
folder_path = r'C:\Users\omkar\Downloads'

def delete_csv_files():
    """Delete only CSV files in the folder."""
    csv_files = glob.glob(os.path.join(folder_path, '*.csv'))
    for file in csv_files:
        try:
            os.remove(file)
            print(f"Deleted: {file}")
        except Exception as e:
            print(f"Error deleting {file}: {e}")

def download_csv(chrome_driver_path, login_url, username, password, csv_file_name):
    """Download the CSV file using a fixed XPath and proper waits."""
    service = Service(chrome_driver_path)
    options = webdriver.ChromeOptions()
    driver = webdriver.Chrome(service=service, options=options)
    
    try:
        print("STARTED")
        driver.get(login_url)
        wait = WebDriverWait(driver, 15)

        # Login
        username_field = wait.until(EC.presence_of_element_located((By.NAME, "email")))
        password_field = driver.find_element(By.NAME, "password")
        username_field.send_keys(username)
        password_field.send_keys(password)
        password_field.send_keys(Keys.RETURN)

        # Wait for page to load post-login
        time.sleep(2)

        # Click the bb_blast_sell_Combined element
        button_xpath = "//b[normalize-space()='bb_blast_sell_Combined']"
        button = wait.until(EC.element_to_be_clickable((By.XPATH, button_xpath)))
        button.click()
        time.sleep(5)

        # Wait for the download link to appear after the above click
        download_button = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//a[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'download csv')]")
            )
        )
        download_button.click()
        time.sleep(5)

        # Check for the downloaded file
        csv_file_path = os.path.join(folder_path, csv_file_name)
        if not os.path.exists(csv_file_path):
            print(f"CSV file not found: {csv_file_path}")
            return []

        # Read and parse the CSV
        stock_list = []
        with open(csv_file_path, mode='r', newline='', encoding='utf-8-sig') as file:
            reader = csv.reader(file)
            header = next(reader)  # Skip header
            for row in reader:
                if len(row) > 0:
                    cell_value = row[0].strip()
                    try:
                        cell_datetime = dt_datetime.strptime(cell_value, '%d-%m-%Y %H:%M')
                        stock_list.append((cell_datetime, row[1]))  # datetime and stock name
                    except ValueError:
                        pass  # Skip parsing errors

        return stock_list

    except Exception as e:
        print(f"An error occurred: {e}")
        return []
    
    finally:
        driver.quit()

# Example usage
chrome_driver_path = r'C:\Users\omkar\AppData\Local\Google\Chrome\User Data\Default\chromedriver-win64\chromedriver.exe'
login_url = "https://chartink.com/login"
username = "om222kore@gmail.com"
password = "GURU222kore#"  
csv_file_name = "Backtest bb_blast_sell_Combined, Technical Analysis Scanner.csv"

# Define the times for the function to run
run_times = [
    dt_datetime.now().replace(hour=9, minute=46, second=15, microsecond=0),
    dt_datetime.now().replace(hour=10, minute=3, second=15, microsecond=0),
    dt_datetime.now().replace(hour=10, minute=46, second=15, microsecond=0),
]

print(f"Automation started at: {dt_datetime.now().strftime('%I:%M:%S %p')}")

# Main loop
for run_time in run_times:
    now = dt_datetime.now()
    if now < run_time:
        wait_time = (run_time - now).total_seconds()
        print(f"Waiting for {wait_time:.2f} seconds until {run_time.strftime('%I:%M:%S %p')}.")
        time.sleep(wait_time)

    print(f"Running at {run_time.strftime('%I:%M:%S %p')}")
    
    delete_csv_files()
    stock_list = download_csv(chrome_driver_path, login_url, username, password, csv_file_name)

    # Get the time exactly 15 minutes before now
    target_time = dt_datetime.now() - timedelta(minutes=15)
    
    # Filter stocks based on the target time
    filtered_stocks = [f"{stock[1]}-EQ" for stock in stock_list if stock[0] == target_time]

    # Print whether stocks were found or not
    if filtered_stocks:
        print(f"Stocks scanned at {target_time.strftime('%I:%M %p')}: {filtered_stocks}")
    else:
        print("No stocks found.")


Automation started at: 09:10:53 AM
Waiting for 2121.24 seconds until 09:46:15 AM.


In [ ]:
# Proper working code to download files at selected intervals using Selenium and Python     #####


import time
import os
import csv
import glob
from datetime import datetime as dt_datetime, timedelta
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Path settings
folder_path = r'C:\Users\omkar\Downloads'
chrome_driver_path = r'C:\Users\omkar\AppData\Local\Google\Chrome\User Data\Default\chromedriver-win64\chromedriver.exe'
login_url = "https://chartink.com/login"
username = "om222kore@gmail.com"
password = "GURU222kore#"
csv_file_path = r"C:\Users\omkar\Downloads\Backtest bb_blast_sell_Combined, Technical Analysis Scanner.csv"
csv_file_name = "Backtest bb_blast_sell_Combined, Technical Analysis Scanner.csv"


# Delete existing CSVs
def delete_csv_files():
    csv_files = glob.glob(os.path.join(folder_path, '*.csv'))
    for file in csv_files:
        try:
            os.remove(file)
            print(f"Deleted: {file}")
        except Exception as e:
            print(f"Error deleting {file}: {e}")

# Round up to next 15-min mark
def round_up_to_next_15_min(dt):
    minutes = (15 - dt.minute % 15) % 15
    if minutes == 0 and dt.second == 0:
        return dt.replace(second=0, microsecond=0)
    return (dt + timedelta(minutes=minutes)).replace(second=0, microsecond=0)

# Download CSV using Selenium
def download_csv(chrome_driver_path, login_url, username, password, csv_file_name):
    service = Service(chrome_driver_path)
    options = webdriver.ChromeOptions()
    driver = webdriver.Chrome(service=service, options=options)
    
    try:
        print("Logging in and downloading CSV...")
        driver.get(login_url)
        wait = WebDriverWait(driver, 15)

        wait.until(EC.presence_of_element_located((By.NAME, "email"))).send_keys(username)
        password_field = driver.find_element(By.NAME, "password")
        password_field.send_keys(password)
        password_field.send_keys(Keys.RETURN)

        time.sleep(2)
        wait.until(EC.element_to_be_clickable((By.XPATH, "//b[normalize-space()='bb_blast_sell_Combined']"))).click()
        time.sleep(5)

        download_button = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//a[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'download csv')]")
            )
        )
        download_button.click()
        time.sleep(5)
    except Exception as e:
        print(f"Download failed: {e}")
    finally:
        driver.quit()

# Parse datetime from string
def parse_datetime(dt_str):
    dt_str = dt_str.strip().lower()
    for fmt in ('%d-%m-%Y %H:%M', '%d-%m-%Y %I:%M %p'):
        try:
            return dt_datetime.strptime(dt_str, fmt)
        except ValueError:
            continue
    raise ValueError(f"Unsupported datetime format: {dt_str}")




def extract_stock_list_from_csv(csv_file_path, target_datetime_str):
    stock_list = []
    try:
        target_datetime = parse_datetime(target_datetime_str)
        with open(csv_file_path, mode='r', encoding='utf-8-sig') as file:
            reader = csv.DictReader(file)
            for row in reader:
                try:
                    row_datetime = parse_datetime(row['date'])
                    if row_datetime == target_datetime:
                        stock_list.append(f"{row['symbol'].strip()}-EQ")
                except ValueError as ve:
                    print(f"Date parsing error: {ve} for row: {row}")
    except Exception as e:
        print(f"Error reading CSV: {e}")
    print("Extracted Stocks:", stock_list)
    return stock_list

# --- Main Execution Block ---

run_times = [
    dt_datetime.now().replace(hour=9, minute=46, second=15, microsecond=0),
    dt_datetime.now().replace(hour=10, minute=3, second=15, microsecond=0),
    dt_datetime.now().replace(hour=10, minute=45, second=15, microsecond=0),
]

print(f"Automation started at: {dt_datetime.now().strftime('%I:%M:%S %p')}")

for run_time in run_times:
    now = dt_datetime.now()
    if now < run_time:
        wait_time = (run_time - now).total_seconds()
        print(f"Waiting for {wait_time:.2f} seconds until {run_time.strftime('%I:%M:%S %p')}.")
        time.sleep(wait_time)

    print(f"\nRunning at {dt_datetime.now().strftime('%I:%M:%S %p')}")

    delete_csv_files()
    stock_list=download_csv(chrome_driver_path, login_url, username, password, csv_file_name)

    # current_time = dt_datetime.now()
    # rounded_time = round_up_to_next_15_min(current_time)
    # target_time = rounded_time - timedelta(minutes=15)
    # formatted_time = target_time.strftime('%d-%m-%Y %I:%M %p')
    
    
    current_time = dt_datetime.now()
    rounded_time = round_up_to_next_15_min(current_time)
    target_time = rounded_time - timedelta(minutes=15)
    formatted_time = target_time.strftime('%d-%m-%Y %H:%M')  # Changed to 24-hour format

    print(f"Current time: {current_time.strftime('%H:%M:%S')}")
    print(f"Rounded to next 15-minute mark: {rounded_time.strftime('%H:%M:%S')}")
    print(f"Target timestamp for filtering: {formatted_time}")


    print(f"Current time: {current_time.strftime('%H:%M:%S')}")
    print(f"Rounded to next 15-minute mark: {rounded_time.strftime('%H:%M:%S')}")
    print(f"Target timestamp for filtering: {formatted_time}")

    
    filtered_stocks = extract_stock_list_from_csv(csv_file_path, formatted_time)

    if filtered_stocks:
        print(f"Stocks found: {filtered_stocks}")
    else:
        print("No stocks found.")


Automation started at: 09:36:18 AM
Waiting for 596.76 seconds until 09:46:15 AM.


In [1]:

# workig

import csv
from datetime import datetime, timedelta
from datetime import datetime

csv_file_path = r"C:\Users\omkar\Downloads\Backtest bb_blast_sell_Combined, Technical Analysis Scanner.csv"

def extract_stock_list_from_csv(csv_file_path, target_datetime_str):
    stock_list = []
    try:
        target_datetime = parse_datetime(target_datetime_str)
        with open(csv_file_path, mode='r', encoding='utf-8-sig') as file:
            reader = csv.DictReader(file)
            for row in reader:
                try:
                    row_datetime = parse_datetime(row['date'])
                    if row_datetime == target_datetime:
                        stock_list.append(f"{row['symbol'].strip()}-EQ")
                except ValueError as ve:
                    print(f"Date parsing error: {ve} for row: {row}")
    except Exception as e:
        print(f"Error reading CSV: {e}")
    print("Extracted Stocks:", stock_list)
    return stock_list


def round_up_to_next_15_min(dt):
    minutes = (15 - dt.minute % 15) % 15
    if minutes == 0 and dt.second == 0:
        return dt.replace(second=0, microsecond=0)
    return (dt + timedelta(minutes=minutes)).replace(second=0, microsecond=0)

# Step 1: Get current time
now = datetime.now()

# Step 2: Round up to next 15-minute interval
rounded = round_up_to_next_15_min(now)

# Step 3: Subtract 15 minutes
final_time = rounded - timedelta(minutes=15)

# Step 4: Format to match CSV format
formatted_time = final_time.strftime('%d-%m-%Y %I:%M %p')

out = extract_stock_list_from_csv(csv_file_path, formatted_time)
print("Output:", out)


Error reading CSV: name 'parse_datetime' is not defined
Extracted Stocks: []
Output: []


In [ ]:
import csv
from datetime import datetime as dt_datetime
import yaml
import pyotp
from NorenRestApiPy.NorenApi import NorenApi

# Shoonya API class
class ShoonyaApiPy(NorenApi):
    def __init__(self):
        super().__init__(host='https://api.shoonya.com/NorenWClientTP/', websocket='wss://api.shoonya.com/NorenWSTP/')

# Initialize API
api = ShoonyaApiPy()
with open('cred.yml') as f:
    cred = yaml.load(f, Loader=yaml.FullLoader)

TOKEN = cred['factor2']
otp = pyotp.TOTP(TOKEN).now()
ret = api.login(
    userid=cred['user'],
    password=cred['pwd'],
    twoFA=otp,
    vendor_code=cred['vc'],
    api_secret=cred['apikey'],
    imei=cred['imei']
)

if ret:
    print("Login Successful")
else:
    print("Login Failed")
    exit()

# Constants
# File path to your CSV (corrected)
CSV_FILE_PATH = r"C:\Users\omkar\Downloads\Backtest bb_blast_sell_Combined, Technical Analysis Scanner.csv"

REMOVE_STOCKS = ['M&M-EQ', 'M&MFIN-EQ', 'J&KBANK-EQ']

# Date parsing helper
def parse_datetime(date_str):
    formats = ["%d-%m-%Y %I:%M %p", "%d-%m-%Y %H:%M"]
    for fmt in formats:
        try:
            return dt_datetime.strptime(date_str, fmt)
        except ValueError:
            continue
    raise ValueError(f"Invalid date format: {date_str}")

# Extract stock list for specific timestamp
def extract_stock_list_from_csv(csv_file_path, target_datetime_str):
    stock_list = []
    try:
        target_datetime = parse_datetime(target_datetime_str)
        with open(csv_file_path, mode='r', encoding='utf-8-sig') as file:
            reader = csv.DictReader(file)
            for row in reader:
                try:
                    row_datetime = parse_datetime(row['date'])
                    if row_datetime == target_datetime:
                        stock_list.append(f"{row['symbol']}-EQ")
                except ValueError:
                    continue
    except Exception as e:
        print(f"Error reading CSV: {e}")
    print("Extracted Stocks:", stock_list)
    return stock_list

# Place market sell orders for stocks
def place_orders(stock_list):
    for symbol in stock_list:
        if symbol in REMOVE_STOCKS:
            continue
        try:
            quote = api.get_quotes(exchange='NSE', token=symbol)
            LTP = float(quote["lp"])
            quantity = round(20000 / LTP)

            api.place_order(
                buy_or_sell='S',
                product_type='I',
                exchange='NSE',
                tradingsymbol=symbol,
                quantity=1,
                discloseqty=0,
                price_type='MKT',
                retention='DAY',
                remarks='AutoSellOrder'
            )
            print(f"Sell order placed for {symbol} | Qty: {quantity} | LTP: {LTP}")
        except Exception as e:
            print(f"Error placing order for {symbol}: {e}")

# Main logic
if __name__ == "__main__":
    target_datetime_str = "29-05-2025 10:00 AM"  # Adjust this as needed
    stocks = extract_stock_list_from_csv(CSV_FILE_PATH, target_datetime_str)
    if stocks:
        place_orders(stocks)
    else:
        print("No stocks to place orders.")


Login Failed
Error reading CSV: [Errno 2] No such file or directory: 'C:\\Users\\omkar\\Downloads\\Backtest bb_blast_sell_Combined, Technical Analysis Scanner.csv'
Extracted Stocks: []
No stocks to place orders.


: 